# Training Optimization and Model Selection - Dzifa, Avi, Erin, Aarabhi

Task #2, part 2. Uses the pipeline from `Model_Setup_InitialTraining.ipynb` for full training runs:

1. GPU environment check
2. Tokenizers for RoBERTa and Legal-BERT (`nlpaueb/legal-bert-base-uncased`)
3. Training with the shared setup (41 labels, `BCEWithLogitsLoss`, Hugging Face `Trainer`)
4. Training and validation loss after every epoch
5. Best epoch and winning run chosen on the validation set only; the test set is scored once, for the final model
6. Best checkpoint, test predictions for Task #4, and comparison with the TF-IDF baseline

**Runs.** Every training run (model + loss) is listed in `RUNS` in the settings cell. Name format for weighted runs: `legal-bert-w-p<power>-c<cap>` (e.g. `legal-bert-w-p0.5-c20`). Add names to `RUNS_TO_RUN` to train them.

**Data variants.** A run can use chunks cut with Legal-BERT's own tokenizer (`data="lb"`, built by the data preparation helpers in section 3) and/or keep only a fraction of the unlabeled training chunks (`keep`). See the settings cell.

**Run modes.** `RUN_MODE = "auto"` runs **full** on a GPU and **smoke** on CPU. Smoke mode uses a small subset to check the pipeline; its scores are meaningless.

**Changes from the initial pipeline:** GPU instead of `use_cpu=True`, validation every epoch, best checkpoint kept, fixed seed.

## 0. Google Colab setup
On Colab: clones the repo, installs `requirements.txt`, mounts Google Drive (results are saved there). Does nothing elsewhere.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1H-contract-review-challenge.git"
REPO_BRANCH = "main"
SAVE_TO_DRIVE = True                          # Colab only: keep results/checkpoints on Google Drive

if IN_COLAB:
    repo_dir = "/content/Accenture-1H-contract-review-challenge"
    if not os.path.exists(repo_dir):
        cloned = subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, repo_dir]).returncode == 0
        if not cloned:   # branch not on GitHub (yet): main has the same data files
            print(f"Branch {REPO_BRANCH} not found, cloning main instead.")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, repo_dir], check=True)
    os.chdir(f"{repo_dir}/notebooks/pipeline")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../../requirements.txt"], check=True)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
print("In Colab:", IN_COLAB, "| working dir:", os.getcwd())

## 1. Setup

In [ ]:
import gc, json, re, shutil, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import transformers
from datasets import Dataset, DatasetDict, Features, Sequence, Value, load_from_disk, disable_progress_bars
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from sklearn.metrics import f1_score, precision_recall_curve

transformers.logging.set_verbosity_error()   # hide the expected "newly initialized classifier" warning
disable_progress_bars()                       # keep tokenization output short
print("torch", torch.__version__, "| transformers", transformers.__version__)

All settings. Add or remove names in `RUNS_TO_RUN` to choose which runs to train in this session (finished runs are skipped).

In [ ]:
RUN_MODE = "auto"      # "auto" (full if a GPU is available, else smoke), "full", or "smoke"
SEED = 42
DATA_DIR = "../../data/processed"

MODELS = {   # short tag -> Hugging Face model name
    "roberta": "roberta-base",
    "legal-bert": "nlpaueb/legal-bert-base-uncased",
}

# Runs. Each entry is one training run; its key is the run name (the folder name under results/ and models/).
#   model       : key of MODELS above
#   power, cap  : weighted loss. pos_weight = clip((negatives / positives) ** power, 1, cap). Leave both out for plain BCE.
# Name format for weighted runs: <model>-w-p<power>-c<cap>, e.g. legal-bert-w-p0.5-c20.
POWERS = [0.5, 0.75]
CAPS = [5, 7, 10, 20]
RUNS = {
    "roberta":             dict(model="roberta"),
    "legal-bert":          dict(model="legal-bert"),
    "legal-bert-weighted": dict(model="legal-bert", power=0.5, cap=10),   # first weighted run (same as legal-bert-w-p0.5-c10)
}
for p in POWERS:
    for c in CAPS:
        if (p, c) != (0.5, 10):
            RUNS[f"legal-bert-w-p{p}-c{c}"] = dict(model="legal-bert", power=p, cap=c)

# Data variants. "orig": chunks cut with RoBERTa's tokenizer (data/processed). "lb": chunks cut with Legal-BERT's own
# tokenizer, so every chunk fills the 512-token window (built by the data preparation helpers on first use, saved to data/processed_lb).
# A run can set data=<variant> and keep=<fraction of the unlabeled TRAINING chunks to keep>. Validation and test are never downsampled.
DATA_VARIANTS = {
    "orig": dict(dir="../../data/processed", cleanup=None),                  # cleanup=None: the model's default (TEXT_CLEANUP)
    "lb":   dict(dir="../../data/processed_lb", chunker="nlpaueb/legal-bert-base-uncased", cleanup=False),
}
# Data experiments: the best weighted loss on other data. Set BEST_LOSS to the winner of the weighted grid.
BEST_LOSS = dict(power=0.5, cap=10)
for d, k in [("orig", 1.0), ("orig", 0.5), ("lb", 1.0), ("lb", 0.5)]:   # orig/1.0 = the baseline, re-run with this notebook so it has contract-level scores
    RUNS[f"legal-bert-w-p{BEST_LOSS['power']}-c{BEST_LOSS['cap']}-d{d}-k{k}"] = dict(model="legal-bert", **BEST_LOSS, data=d, keep=k)
assert all(r["model"] in MODELS and r.get("data", "orig") in DATA_VARIANTS for r in RUNS.values())

# Which runs to train in this session (add names from RUNS, e.g. "legal-bert-w-p0.5-c20").
RUNS_TO_RUN = ["roberta", "legal-bert", "legal-bert-weighted"]
SKIP_FINISHED = True   # don't retrain a model whose summary.json already exists

MAX_LENGTH = 512       # both models' input limit
OVERLAP = 128          # chunk overlap used in Subword_Tokenization_Chunking.ipynb
THRESHOLD = 0.5        # fixed here; per-category thresholds are tuned in Task #3
SELECT_METRIC = "macro_ap"   # validation metric used to choose the best epoch of each run (see section 5)
WINNER_METRIC = "macro_f1"   # validation metric for the winning run: "macro_f1" (project goal) or "macro_ap". Use "contract_macro_f1" when
                             # comparing different chunkings (data="lb" vs "orig"): chunk-level scores are not comparable across chunkings
SCORE_TEST_PER_RUN = False   # False: the test set is NOT scored during experiments; it is scored once, for FINAL_RUN (section 7)
FINAL_RUN = None             # when experiments are done, set this to the winning run name (section 7 prints it)

FULL = dict(num_epochs=15, learning_rate=2e-5, effective_batch_size=16, max_length=MAX_LENGTH,
            warmup_ratio=0.1, weight_decay=0.01, subset=None)
SMOKE = dict(num_epochs=3, learning_rate=5e-5, effective_batch_size=16, max_length=128,
             warmup_ratio=0.1, weight_decay=0.01, subset={"train": 384, "validation": 128, "test": 128})

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"{DATA_DIR} not found. Run this notebook from the notebooks/pipeline/ folder.")
set_seed(SEED)

## 2. GPU training environment
Checks the device and turns on mixed precision on GPU (bf16 if supported, else fp16). The per-step batch size depends on GPU memory; gradient accumulation keeps the **effective batch size at 16**.

In [ ]:
HAS_GPU = torch.cuda.is_available()
MODE = ("full" if HAS_GPU else "smoke") if RUN_MODE == "auto" else RUN_MODE
CFG = dict(FULL if MODE == "full" else SMOKE)

print("Python:", platform.python_version(), "| CPU cores:", os.cpu_count())
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name} | {vram_gb:.1f} GB | CUDA {torch.version.cuda}")
    USE_BF16 = torch.cuda.is_bf16_supported()
    USE_FP16 = not USE_BF16
    per_device = 16 if vram_gb >= 40 else 8 if vram_gb >= 14 else 4
else:
    print("GPU: none found. Training runs on CPU.")
    if IN_COLAB:
        print("  -> On Colab: Runtime > Change runtime type > T4 GPU (or better), then run all again.")
    USE_BF16 = USE_FP16 = False
    per_device = 8
if MODE == "full" and not HAS_GPU:
    print("WARNING: full mode on CPU will take many hours per epoch.")

# Make sure the device can actually run a computation
device = torch.device("cuda" if HAS_GPU else "cpu")
a = torch.randn(1024, 1024, device=device)
assert torch.isfinite(a @ a).all()

CFG["per_device_batch_size"] = min(per_device, CFG["effective_batch_size"])
CFG["grad_accum_steps"] = CFG["effective_batch_size"] // CFG["per_device_batch_size"]
print(f"Device check passed on {device}. Mode: {MODE} | precision: "
      f"{'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32'} | batch {CFG['per_device_batch_size']} "
      f"x {CFG['grad_accum_steps']} accumulation = {CFG['effective_batch_size']}")

In [ ]:
# results/ is committed, models/ is not. Each run gets its own folder (e.g. 15ep). Smoke runs stay under models/.
RUN_NAME = f"{CFG['num_epochs']}ep"
OUTPUT_ROOT = "/content/drive/MyDrive/Accenture-1H-contract-review-challenge" if (IN_COLAB and SAVE_TO_DRIVE) else "../.."
RESULTS_DIR = f"{OUTPUT_ROOT}/results/training/{RUN_NAME}" if MODE == "full" else f"{OUTPUT_ROOT}/models/training_smoke/results"
MODELS_DIR = f"{OUTPUT_ROOT}/models/training/{RUN_NAME}" if MODE == "full" else f"{OUTPUT_ROOT}/models/training_smoke"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("Results:", RESULTS_DIR, "\nCheckpoints:", MODELS_DIR)

In [ ]:
# Bring in finished runs that are committed to the repo (results/), so the comparison in section 7 includes them.
# Checkpoints are not in git: scoring a run on the test set needs the Drive that trained it.
for r in RUNS:
    committed, here = f"../../results/training/{RUN_NAME}/{r}", f"{RESULTS_DIR}/{r}"
    if os.path.exists(committed) and not os.path.exists(here):
        shutil.copytree(committed, here)
print(sorted(os.listdir(RESULTS_DIR)))

## 3. Load the chunks and the validation split
Same inputs and 61-contract validation split (`val_titles.json`) as `Transformer_Dataset_Prep.ipynb`. The splits are rebuilt from the text because Legal-BERT needs its own tokenizer; `roberta_dataset/` only has RoBERTa token IDs.

In [ ]:
train_df = pd.read_parquet(f"{DATA_DIR}/train_chunks_labeled.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test_chunks_labeled.parquet")
with open(f"{DATA_DIR}/categories.json") as f:
    CATEGORIES = json.load(f)
with open(f"{DATA_DIR}/val_titles.json") as f:
    val_titles = set(json.load(f)["val_titles"])
train_titles = set(train_df["title"]) - val_titles

FEATURES = Features({"title": Value("string"), "chunk_id": Value("int64"), "chunk_text": Value("string"),
                     "labels": Sequence(Value("float32"), length=len(CATEGORIES))})

def build_split(df, titles):
    part = df[df["title"].isin(titles)].reset_index(drop=True)
    out = part[["title", "chunk_id", "chunk_text"]].copy()
    out["labels"] = list(part[CATEGORIES].to_numpy(dtype=np.float32))
    return Dataset.from_pandas(out, features=FEATURES, preserve_index=False)

raw = DatasetDict({"train": build_split(train_df, train_titles),
                   "validation": build_split(train_df, val_titles),
                   "test": build_split(test_df, set(test_df["title"]))})

assert len(CATEGORIES) == 41 and len(val_titles) == 61 and len(train_titles) == 347
assert not (val_titles & train_titles) and not (set(train_df["title"]) & set(test_df["title"]))
assert (raw["train"].num_rows, raw["validation"].num_rows, raw["test"].num_rows) == (12112, 2069, 2898)
print({k: v.num_rows for k, v in raw.items()}, "| load checks passed.")

### Data variants
`get_raw(data, keep)` returns the train / validation / test chunks of a data variant. The validation contracts are the same in every variant (`val_titles.json`). Only the **training** split is downsampled (`keep` = fraction of chunks without any label that are kept); validation and test keep every chunk.

### Data preparation helpers
Functions to cut the contracts into overlapping chunks with any tokenizer, label the chunks (a chunk gets a 1 for a category when a lawyer-highlighted answer overlaps it, as in `03_data_labeling.ipynb`), check the labels, keep only a fraction of the unlabeled training chunks (`downsample_unlabeled`), and score at contract level (`contract_level_scores`, which does not depend on how the contracts were chunked). Used by the data variants below.
Use transformers 4.57.x (see `requirements.txt`): version 5 did not return all overflow chunks.

In [ ]:
# Data preparation helpers (chunk, label, validate, downsample, contract-level scores). Same logic as 02_data_tokenization_and_chunking and 03_data_labeling.
import json, os
def get_category(question):
    match = re.search(r'"([^"]+)"', question)
    return match.group(1) if match else None


def get_contracts(cuad_json):
    rows = [{"title": c["title"], "context": p["context"]} for c in cuad_json["data"] for p in c["paragraphs"]]
    return pd.DataFrame(rows)


def get_answer_spans(cuad_json):
    rows = []
    for c in cuad_json["data"]:
        for p in c["paragraphs"]:
            for qa in p["qas"]:
                for a in qa["answers"]:
                    rows.append({"title": c["title"], "category": get_category(qa["question"]),
                                 "answer_start": a["answer_start"],
                                 "answer_end": a["answer_start"] + len(a["text"]), "answer_text": a["text"]})
    return pd.DataFrame(rows)


def get_categories(cuad_json):
    return sorted({get_category(qa["question"]) for c in cuad_json["data"] for p in c["paragraphs"] for qa in p["qas"]})


def chunk_contracts(contracts, tokenizer, max_length=MAX_LENGTH, overlap=OVERLAP):
    """Split every contract into overlapping chunks of at most max_length tokens of `tokenizer`.

    Returns title, chunk_id, char_start, char_end, chunk_text (positions in the original contract text).
    """
    rows = []
    for title, context in zip(contracts["title"], contracts["context"]):
        enc = tokenizer(context, max_length=max_length, stride=overlap, truncation=True,
                        return_overflowing_tokens=True, return_offsets_mapping=True)
        for chunk_id, offsets in enumerate(enc["offset_mapping"]):
            valid = [(s, e) for s, e in offsets if e > s]      # drops [CLS], [SEP] and padding
            rows.append({"title": title, "chunk_id": chunk_id, "char_start": valid[0][0],
                         "char_end": valid[-1][1], "chunk_text": context[valid[0][0]:valid[-1][1]]})
    return pd.DataFrame(rows)


def label_chunks(chunks_df, spans_df, categories):
    """Same rule as Labeling.ipynb: label 1 if any answer of the category overlaps the chunk."""
    labeled = chunks_df.reset_index(drop=True).copy()
    labels = np.zeros((len(labeled), len(categories)), dtype=np.int8)
    cat_index = {c: i for i, c in enumerate(categories)}
    chunk_rows = labeled.groupby("title").indices
    for title, spans in spans_df.groupby("title"):
        if title not in chunk_rows:
            continue
        rows = chunk_rows[title]
        starts = labeled.loc[rows, "char_start"].to_numpy()
        ends = labeled.loc[rows, "char_end"].to_numpy()
        for _, span in spans.iterrows():
            overlaps = (starts < span["answer_end"]) & (ends > span["answer_start"])
            labels[rows[overlaps], cat_index[span["category"]]] = 1
    return pd.concat([labeled, pd.DataFrame(labels, columns=categories)], axis=1)


def validate_labels(labeled, spans, contracts, categories, name):
    """The checks from Labeling.ipynb, plus: every answer is covered by a positive chunk."""
    assert labeled[categories].isin([0, 1]).all().all(), f"{name}: labels other than 0/1"
    assert spans["category"].isin(categories).all(), f"{name}: unknown category"
    assert set(contracts["title"]) == set(labeled["title"]), f"{name}: a contract has no chunks"
    assert (labeled["char_start"] < labeled["char_end"]).all(), f"{name}: empty or reversed chunk"
    text = dict(zip(contracts["title"], contracts["context"]))
    ok = [text[t][s:e] == c for t, s, e, c in zip(labeled["title"], labeled["char_start"], labeled["char_end"], labeled["chunk_text"])]
    assert all(ok), f"{name}: chunk_text does not match the contract text"
    by_title = {t: g for t, g in labeled.groupby("title")}
    for t, cat, a0, a1 in zip(spans["title"], spans["category"], spans["answer_start"], spans["answer_end"]):
        g = by_title[t]
        assert ((g["char_start"] < a1) & (g["char_end"] > a0) & (g[cat] == 1)).any(), f"{name}: answer not covered ({t}, {cat})"
    print(f"{name}: all checks passed ({len(labeled):,} chunks, {len(spans):,} answers)")


def downsample_unlabeled(df, categories, keep, seed=42):
    """Keep every chunk that has at least one label, and a random `keep` fraction of the chunks with none.

    Use on the TRAINING chunks only. Validation and test keep all chunks, so scores stay comparable.
    """
    if keep >= 1.0:
        return df.reset_index(drop=True)
    unlabeled = df[categories].to_numpy().sum(axis=1) == 0
    rng = np.random.default_rng(seed)
    keep_mask = ~unlabeled | (rng.random(len(df)) < keep)
    return df[keep_mask].reset_index(drop=True)


def contract_level_scores(titles, y_true, probs, threshold=0.5):
    """Contract-level micro/macro F1. A clause counts as present in a contract if any of its chunks is labeled
    (gold) or predicted above the threshold. This does not depend on how the contract was cut into chunks, so it
    is the fair way to compare two chunkings."""
    from sklearn.metrics import precision_recall_fscore_support
    y_true, probs = np.asarray(y_true), np.asarray(probs)
    df = pd.DataFrame(np.hstack([y_true, probs >= threshold]).astype(int))
    df["title"] = list(titles)
    n = y_true.shape[1]
    agg = df.groupby("title").max()
    gold, pred = agg.iloc[:, :n].to_numpy(), agg.iloc[:, n:2 * n].to_numpy()
    has = gold.sum(axis=0) > 0
    out = {}
    for avg in ["micro", "macro"]:
        p, r, f, _ = precision_recall_fscore_support(gold[:, has], pred[:, has], average=avg, zero_division=0)
        out.update({f"{avg}_precision": p, f"{avg}_recall": r, f"{avg}_f1": f})
    out["contracts"] = int(len(agg))
    return out


def length_report(chunks, tokenizer, max_length=MAX_LENGTH):
    """Re-tokenize chunk_text alone and report how many chunks exceed max_length."""
    lengths = np.array([len(x) for x in tokenizer(list(chunks["chunk_text"]), verbose=False)["input_ids"]])
    return {"chunks": len(chunks), "median_tokens": int(np.median(lengths)), "max_tokens": int(lengths.max()),
            "over_limit": int((lengths > max_length).sum())}


def build_variant(tokenizer, out_dir, train_json_path="../../data/train_separate_questions.json",
                  test_json_path="../../data/test.json", max_length=MAX_LENGTH, overlap=OVERLAP):
    """Chunk and label the train and test contracts with `tokenizer` and save them to out_dir."""
    with open(train_json_path) as f:
        train_json = json.load(f)
    with open(test_json_path) as f:
        test_json = json.load(f)
    categories = get_categories(train_json)
    assert len(categories) == 41
    os.makedirs(out_dir, exist_ok=True)
    for name, data in (("train", train_json), ("test", test_json)):
        contracts, spans = get_contracts(data), get_answer_spans(data)
        chunks = chunk_contracts(contracts, tokenizer, max_length, overlap)
        labeled = label_chunks(chunks, spans, categories)
        validate_labels(labeled, spans, contracts, categories, name.upper())
        print(name, "length check on re-tokenized chunk text:", length_report(chunks, tokenizer, max_length))
        labeled.to_parquet(f"{out_dir}/{name}_chunks_labeled.parquet", index=False)
    with open(f"{out_dir}/categories.json", "w") as f:
        json.dump(categories, f, indent=1)
    print("Saved to", out_dir)


In [ ]:
RAW_CACHE = {("orig", 1.0): raw}

def ensure_variant(data):
    # Build the chunks of a data variant the first time it is needed (a minute or two), then reuse the saved files.
    cfg = DATA_VARIANTS[data]
    if os.path.exists(f"{cfg['dir']}/train_chunks_labeled.parquet"):
        return
    chunker = AutoTokenizer.from_pretrained(cfg["chunker"], use_fast=True)
    build_variant(chunker, cfg["dir"], "../../data/train_separate_questions.json", "../../data/test.json")

def get_raw(data="orig", keep=1.0):
    if (data, keep) not in RAW_CACHE:
        if data != "orig":
            ensure_variant(data)
        d = DATA_VARIANTS[data]["dir"]
        tr_df, te_df = (train_df, test_df) if data == "orig" else (pd.read_parquet(f"{d}/train_chunks_labeled.parquet"),
                                                                    pd.read_parquet(f"{d}/test_chunks_labeled.parquet"))
        train_part = downsample_unlabeled(tr_df[tr_df["title"].isin(train_titles)], CATEGORIES, keep, SEED)
        RAW_CACHE[(data, keep)] = DatasetDict({"train": build_split(train_part, train_titles),
                                               "validation": build_split(tr_df, val_titles),
                                               "test": build_split(te_df, set(te_df["title"]))})
    return RAW_CACHE[(data, keep)]

print("Data variants:", ", ".join(f"{k} ({v['dir']})" for k, v in DATA_VARIANTS.items()))

## 4. Tokenizers: RoBERTa and Legal-BERT
- **RoBERTa** (`roberta-base`): cased byte-level BPE. Chunks were cut to 512 RoBERTa tokens.
- **Legal-BERT**: uncased WordPiece pretrained on legal text, so token counts differ.

**Punctuation cleanup for Legal-BERT.** Legal-BERT splits table borders (`--------`) into one token per dash, so some chunks exceed 1,700 tokens. Runs of 3+ identical punctuation marks are shortened to 3 before Legal-BERT tokenization. RoBERTa gets the original text. Apply the same cleanup at prediction time (noted in `training_info.json`).

In [ ]:
PUNCT_RUN = re.compile(r"([^\w\s])\1{2,}")
TEXT_CLEANUP = {"roberta": False, "legal-bert": True}

def clean_text(text):
    return PUNCT_RUN.sub(r"\1\1\1", text)

tokenizers = {tag: AutoTokenizer.from_pretrained(MODELS[tag], use_fast=True) for tag in MODELS}

def tokenize_split(ds, tag, max_length, cleanup=None):
    tok = tokenizers[tag]
    cleanup = TEXT_CLEANUP[tag] if cleanup is None else cleanup
    def fn(batch):
        texts = [clean_text(t) for t in batch["chunk_text"]] if cleanup else batch["chunk_text"]
        return tok(texts, max_length=max_length, truncation=True)
    return ds.map(fn, batched=True, remove_columns=["chunk_text"])

for tag, tok in tokenizers.items():
    print(f"{tag:11s} {type(tok).__name__:20s} vocab {tok.vocab_size:,} | lowercase: {bool(getattr(tok, 'do_lower_case', False))} "
          f"| special: {tok.cls_token} ... {tok.sep_token}")

example = "This Agreement shall be governed by the laws of the State of New York."
for tag, tok in tokenizers.items():
    print(f"\n{tag}: {tok.tokenize(example)}")

### How each tokenizer fits our chunks
Cuts of up to 128 tokens are safe because the next chunk repeats them (overlap). Cuts on a contract's last chunk, or longer than the overlap, lose text.

In [ ]:
def length_report(tag):
    tok, rows = tokenizers[tag], []
    for name, ds in raw.items():
        texts = [clean_text(t) for t in ds["chunk_text"]] if TEXT_CLEANUP[tag] else list(ds["chunk_text"])
        lengths = np.array([len(x) for x in tok(texts, verbose=False)["input_ids"]])
        dropped = np.maximum(lengths - MAX_LENGTH, 0)
        chunks = ds.select_columns(["title", "chunk_id"]).to_pandas()
        is_last = (chunks["chunk_id"] == chunks.groupby("title")["chunk_id"].transform("max")).to_numpy()
        rows.append({"model": tag, "split": name, "chunks": len(ds), "median_tokens": int(np.median(lengths)),
                     "max_tokens": int(lengths.max()), "truncated": int((dropped > 0).sum()),
                     "cut_past_overlap": int((dropped > OVERLAP).sum()), "last_chunk_cut": int((dropped[is_last] > 0).sum()),
                     "tokens_lost_%": round(100 * dropped.sum() / lengths.sum(), 2)})
    return rows

lengths_df = pd.DataFrame(length_report("roberta") + length_report("legal-bert"))
lengths_df

In [ ]:
# RoBERTa: same as Transformer_Dataset_Prep.ipynb (at most 3 tokens cut, never a last chunk)
rob = lengths_df[lengths_df["model"] == "roberta"]
assert rob["max_tokens"].max() <= MAX_LENGTH + 3 and rob["last_chunk_cut"].sum() == 0
# Legal-BERT: the punctuation cleanup keeps the loss small
lb = lengths_df[lengths_df["model"] == "legal-bert"]
assert lb["tokens_lost_%"].max() < 1.0

# If roberta_dataset/ exists (built by Transformer_Dataset_Prep.ipynb), our RoBERTa tokens must match it exactly
roberta_full = DatasetDict({k: tokenize_split(v, "roberta", MAX_LENGTH) for k, v in raw.items()})
if os.path.exists(f"{DATA_DIR}/roberta_dataset"):
    prepared = load_from_disk(f"{DATA_DIR}/roberta_dataset")
    for k in raw:
        assert prepared[k]["title"] == roberta_full[k]["title"] and prepared[k]["chunk_id"] == roberta_full[k]["chunk_id"]
        assert prepared[k]["input_ids"] == roberta_full[k]["input_ids"] and prepared[k]["labels"] == roberta_full[k]["labels"]
    del prepared
    print("RoBERTa tokens match data/processed/roberta_dataset exactly.")
else:
    print("roberta_dataset/ not found (not in git); skipped the match check.")
del roberta_full
print("Tokenizer checks passed.")

**Result.** RoBERTa: about 11% of chunks lose 1–3 tokens, all within the overlap. Legal-BERT: most chunks are shorter, about 8% still exceed 512, and under 0.5% of tokens are lost after the cleanup.

## 5. Training setup
**Model and loss.** `AutoModelForSequenceClassification` with 41 outputs and `problem_type="multi_label_classification"` (`BCEWithLogitsLoss`). Plain BCE by default. A run in `RUNS` can set `power` and `cap` to weight rare categories (pos_weight = clip((negatives / positives) ** power, 1, cap)). Per-category thresholds did not help (sections 10 and 12).

**Best epoch = highest validation macro average precision (AP).** AP doesn't depend on a threshold, and macro averaging weights rare categories equally. Validation loss and F1 at 0.5 are reported too. Categories with no positives in a split are excluded (as in the baseline).

**Winning run = highest validation macro F1 (`WINNER_METRIC`).** The test set is never used to choose an epoch or a run. It is scored once, for the winner (`FINAL_RUN`, end of section 7).

**Comparing chunkings.** Chunk-level scores are not comparable between `orig` and `lb` chunks (different chunks, different count). Contract-level scores are (a clause counts as present if any chunk is labeled, or predicted above 0.5). Set `WINNER_METRIC = "contract_macro_f1"` to choose between data variants.

In [ ]:
def multilabel_scores(y_true, probs, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    has_pos = y_true.sum(axis=0) > 0
    yt, pr = y_true[:, has_pos], probs[:, has_pos]
    yp = (pr >= threshold).astype(int)
    scores = {"macro_ap": average_precision_score(yt, pr, average="macro"),
              "micro_ap": average_precision_score(yt, pr, average="micro")}
    for avg in ["micro", "macro"]:
        p, r, f, _ = precision_recall_fscore_support(yt, yp, average=avg, zero_division=0)
        scores.update({f"{avg}_precision": p, f"{avg}_recall": r, f"{avg}_f1": f})
    scores["categories_scored"] = int(has_pos.sum())
    return scores

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return multilabel_scores(labels, sigmoid(logits))

# Quick check on made-up data: perfect predictions score 1, and a category with no positives is skipped
y = np.array([[1, 0, 0], [0, 1, 0], [1, 1, 0]])
s = multilabel_scores(y, y.astype(float))
assert s["macro_ap"] == 1 and s["micro_f1"] == 1 and s["categories_scored"] == 2
print("Metric checks passed.")

In [ ]:
def prepare_splits(tag, data="orig", keep=1.0):
    # Tokenized train / validation / test for one model and data variant. Smoke mode uses a fixed random subset.
    splits = {}
    for name, ds in get_raw(data, keep).items():
        if CFG["subset"]:
            ds = ds.shuffle(seed=SEED).select(range(CFG["subset"][name]))
        splits[name] = tokenize_split(ds, tag, CFG["max_length"], DATA_VARIANTS[data].get("cleanup"))
    return splits

def training_args(out_dir):
    return TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=CFG["num_epochs"],
        learning_rate=CFG["learning_rate"],
        per_device_train_batch_size=CFG["per_device_batch_size"],
        per_device_eval_batch_size=CFG["per_device_batch_size"] * 2,
        gradient_accumulation_steps=CFG["grad_accum_steps"],
        warmup_ratio=CFG["warmup_ratio"],
        weight_decay=CFG["weight_decay"],
        eval_strategy="epoch",            # validation loss + scores after every epoch
        logging_strategy="epoch",         # training loss averaged over each epoch
        save_strategy="epoch",
        save_total_limit=1,               # keeps the best checkpoint (and the latest one)
        save_only_model=True,             # no optimizer state: ~0.5 GB per checkpoint instead of ~1.5 GB
        load_best_model_at_end=True,
        metric_for_best_model=SELECT_METRIC,
        greater_is_better=True,
        bf16=USE_BF16, fp16=USE_FP16,
        dataloader_num_workers=2 if HAS_GPU else 0,
        dataloader_pin_memory=HAS_GPU,
        seed=SEED, data_seed=SEED,
        report_to="none",
        disable_tqdm=not HAS_GPU,         # keep the saved CPU smoke run readable
    )

## 6. Train
For each run in `RUNS_TO_RUN`: train for `num_epochs` with validation after each epoch, reload the best epoch, then save:
- checkpoint and tokenizer to `models/training/<run>/<run name>/best/`
- history, per-category validation scores and summary to `results/training/<run>/<run name>/`

`<run>` is the epoch count (e.g. `15ep`), so new runs don't overwrite old ones. With `SKIP_FINISHED`, finished runs are skipped. Test predictions are written when the test set is scored (see `SCORE_TEST_PER_RUN` and section 7).

In [ ]:
def epoch_history(log_history):
    train = {round(h["epoch"]): h["loss"] for h in log_history if "loss" in h and "eval_loss" not in h}
    rows = []
    for h in log_history:
        if "eval_loss" in h:
            e = round(h["epoch"])
            rows.append({"epoch": e, "train_loss": train.get(e), "val_loss": h["eval_loss"],
                         **{k[5:]: v for k, v in h.items() if k.startswith("eval_") and k[5:] in
                            ["macro_ap", "micro_ap", "micro_f1", "macro_f1", "micro_precision", "micro_recall"]}})
    return pd.DataFrame(rows)


In [ ]:
def compute_pos_weight(power, cap, data="orig", keep=1.0):
    # One weight per category: (negatives / positives) ** power, clipped to [1, cap]. Uses the full training split of the
    # data variant (after downsampling, so downsampling the unlabeled chunks also lowers the weights).
    labels = np.array(get_raw(data, keep)["train"]["labels"])
    positives = labels.sum(axis=0)
    negatives = len(labels) - positives
    weights = np.clip((negatives / np.maximum(positives, 1.0)) ** power, 1.0, cap)
    return torch.tensor(weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Ensure pos_weight is on the same device and precision as logits
        pos_weight = self.pos_weight.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

for name in RUNS_TO_RUN:
    if RUNS[name].get("power") is not None:
        w = compute_pos_weight(RUNS[name]["power"], RUNS[name]["cap"], RUNS[name].get("data", "orig"), RUNS[name].get("keep", 1.0))
        print(f"{name}: pos_weight min / median / max = {w.min():.2f} / {w.median():.2f} / {w.max():.2f}")


In [ ]:

def score_test_once(name):
    # Scores the saved best checkpoint of one run on the test set and writes its test predictions (Task #4).
    # Do this once, for the final model only.
    tag = RUNS[name]["model"]
    res_dir, model_dir = f"{RESULTS_DIR}/{name}", f"{MODELS_DIR}/{name}"
    with open(f"{res_dir}/summary.json") as f:
        summary = json.load(f)
    if "test_at_best_epoch" in summary:
        print(f"[{name}] test already scored (macro F1 {summary['test_at_best_epoch']['macro_f1']:.3f}). Not scoring it again.")
        return
    data = RUNS[name].get("data", "orig")
    ds = get_raw(data, 1.0)["test"]
    if CFG["subset"]:
        ds = ds.shuffle(seed=SEED).select(range(CFG["subset"]["test"]))
    test = tokenize_split(ds, tag, CFG["max_length"], DATA_VARIANTS[data].get("cleanup"))
    cols = ["input_ids", "attention_mask", "labels"]
    model = AutoModelForSequenceClassification.from_pretrained(f"{model_dir}/best")
    args = TrainingArguments(output_dir=f"{model_dir}/test_tmp", per_device_eval_batch_size=CFG["per_device_batch_size"] * 2,
                             bf16=USE_BF16, fp16=USE_FP16, report_to="none", disable_tqdm=not HAS_GPU)
    trainer = Trainer(model=model, args=args, data_collator=DataCollatorWithPadding(tokenizers[tag]))
    test_probs = sigmoid(trainer.predict(test.select_columns(cols)).predictions)

    # One row per chunk, one probability per category
    preds = pd.DataFrame(test_probs, columns=CATEGORIES)
    preds.insert(0, "chunk_id", test["chunk_id"])
    preds.insert(0, "title", test["title"])
    preds.to_parquet(f"{res_dir}/test_predictions.parquet", index=False)
    if data != "orig" and not CFG["subset"]:      # other chunkings: save where each chunk sits in its contract
        pd.read_parquet(f"{DATA_VARIANTS[data]['dir']}/test_chunks_labeled.parquet")[["title", "chunk_id", "char_start", "char_end"]] \
            .to_parquet(f"{res_dir}/test_chunks.parquet", index=False)
    scores = multilabel_scores(np.array(test["labels"]), test_probs)
    summary["test_at_best_epoch"] = {k: float(v) for k, v in scores.items()}
    summary["test_contract"] = {k: float(v) for k, v in contract_level_scores(test["title"], test["labels"], test_probs).items()}
    with open(f"{res_dir}/summary.json", "w") as f:
        json.dump(summary, f, indent=1)
    print(f"[{name}] test (scored once): micro F1 {scores['micro_f1']:.3f}, macro F1 {scores['macro_f1']:.3f}")
    del trainer, model
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

def run_experiment(name):
    run = RUNS[name]
    tag = run["model"]
    weighted = run.get("power") is not None
    data, keep = run.get("data", "orig"), run.get("keep", 1.0)
    res_dir, model_dir = f"{RESULTS_DIR}/{name}", f"{MODELS_DIR}/{name}"
    if SKIP_FINISHED and os.path.exists(f"{res_dir}/summary.json"):
        print(f"[{name}] already finished, skipping (set SKIP_FINISHED = False to retrain)")
        return
    os.makedirs(res_dir, exist_ok=True)
    set_seed(SEED)
    splits = prepare_splits(tag, data, keep)
    cols = ["input_ids", "attention_mask", "labels"]
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[tag], num_labels=len(CATEGORIES), problem_type="multi_label_classification",
        id2label=dict(enumerate(CATEGORIES)), label2id={c: i for i, c in enumerate(CATEGORIES)})
    trainer_args = dict(model=model, args=training_args(f"{model_dir}/checkpoints"),
                        train_dataset=splits["train"].select_columns(cols),
                        eval_dataset=splits["validation"].select_columns(cols),
                        data_collator=DataCollatorWithPadding(tokenizers[tag]),
                        compute_metrics=compute_metrics)
    if weighted:
        trainer = WeightedTrainer(**trainer_args, pos_weight=compute_pos_weight(run["power"], run["cap"], data, keep))
    else:
        trainer = Trainer(**trainer_args)

    print(f"[{name}] training {MODELS[tag]} on {splits['train'].num_rows:,} chunks for {CFG['num_epochs']} epochs ...")
    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60

    history = epoch_history(trainer.state.log_history)
    best = history.loc[history[SELECT_METRIC].idxmax()]      # best epoch: validation only
    best_epoch = int(best["epoch"])
    history.to_csv(f"{res_dir}/epoch_history.csv", index=False)

    # Best checkpoint (load_best_model_at_end already put it back into trainer.model)
    best_dir = f"{model_dir}/best"
    trainer.save_model(best_dir)
    tokenizers[tag].save_pretrained(best_dir)

    # Per-category validation scores of the best epoch
    val_probs = sigmoid(trainer.predict(splits["validation"].select_columns(cols)).predictions)
    y_val = np.array(splits["validation"]["labels"])
    pd.DataFrame({"val_ap": [average_precision_score(y_val[:, j], val_probs[:, j]) if y_val[:, j].any() else np.nan
                             for j in range(len(CATEGORIES))],
                  "val_positives": y_val.sum(0).astype(int)}, index=CATEGORIES).to_csv(f"{res_dir}/val_per_category.csv")

    # Contract-level validation scores: comparable across chunkings (unlike chunk-level scores)
    val_contract = {k: float(v) for k, v in contract_level_scores(splits["validation"]["title"], y_val, val_probs).items()}

    summary = {"model": MODELS[tag], "tag": tag, "run": name, "mode": MODE, "data": data, "keep": keep, "best_epoch": best_epoch,
               "select_metric": SELECT_METRIC, "best_val": {k: float(v) for k, v in best.items() if k != "epoch"},
               "val_contract": val_contract,
               "loss": {"type": "weighted_bce", "power": run["power"], "cap": run["cap"]} if weighted else {"type": "bce"},
               "train_minutes": round(minutes, 1),
               "device": torch.cuda.get_device_name(0) if HAS_GPU else "cpu",
               "config": CFG, "text_cleanup": TEXT_CLEANUP[tag],
               "train_chunks": splits["train"].num_rows, "checkpoint": best_dir}
    with open(f"{best_dir}/training_info.json", "w") as f:
        json.dump({**summary, "categories": CATEGORIES,
                   "note": "apply clean_text() (runs of 3+ identical punctuation -> 3) before tokenizing" if TEXT_CLEANUP[tag] else ""}, f, indent=1)
    with open(f"{res_dir}/summary.json", "w") as f:
        json.dump(summary, f, indent=1)

    print(f"[{name}] done in {minutes:.1f} min. Best epoch: {best_epoch} (val {SELECT_METRIC} {best[SELECT_METRIC]:.3f}, val macro F1 {best['macro_f1']:.3f}, contract-level {val_contract['macro_f1']:.3f})")
    del trainer, model
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()
    if SCORE_TEST_PER_RUN:
        score_test_once(name)

for name in RUNS_TO_RUN:
    run_experiment(name)

## 7. Compare epochs and models
### Loss per epoch

In [ ]:
summaries, histories = {}, {}
for tag in RUNS:
    if os.path.exists(f"{RESULTS_DIR}/{tag}/summary.json"):
        with open(f"{RESULTS_DIR}/{tag}/summary.json") as f:
            summaries[tag] = json.load(f)
        histories[tag] = pd.read_csv(f"{RESULTS_DIR}/{tag}/epoch_history.csv")
assert summaries, "No finished runs found. Run section 6 first."

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
PALETTE = ["#2a78d6", "#eb6834", "#1a9a6c", "#8a5cd0", "#c9a227", "#d6457a", "#2f9fb0", "#7a7a7a", "#a0522d", "#3b5bdb"]
COLORS = {tag: PALETTE[i % len(PALETTE)] for i, tag in enumerate(RUNS)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for tag, h in histories.items():
    axes[0].plot(h["epoch"], h["train_loss"], marker="o", color=COLORS[tag], label=f"{tag} train")
    axes[0].plot(h["epoch"], h["val_loss"], marker="o", linestyle="--", color=COLORS[tag], label=f"{tag} validation")
    axes[1].plot(h["epoch"], h[SELECT_METRIC], marker="o", color=COLORS[tag], label=tag)
    b = summaries[tag]["best_epoch"]
    axes[1].scatter([b], [h.loc[h["epoch"] == b, SELECT_METRIC].iloc[0]], s=160, facecolors="none",
                    edgecolors=COLORS[tag], linewidths=2, zorder=3)
axes[0].set_title("Training vs validation loss", loc="left", color=INK)
axes[0].set_ylabel("BCE loss")
axes[1].set_title(f"Validation {SELECT_METRIC} (circle = best epoch)", loc="left", color=INK)
axes[1].set_ylabel(SELECT_METRIC)
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(sorted({e for h in histories.values() for e in h["epoch"]}))
    ax.grid(axis="y", color=GRID)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED)
if MODE == "smoke":
    fig.suptitle("SMOKE RUN (CPU, tiny subset): checks the pipeline only, the numbers are not meaningful", color="#b3261e", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### Validation scores per epoch

In [ ]:
table = pd.concat({tag: h.set_index("epoch") for tag, h in histories.items()}, names=["model", "epoch"])
table.round(4)

### Best run and epoch
Chosen on validation only (`WINNER_METRIC`). The test columns stay empty until the test set is scored, once, for the winner (last cell of this section). TF-IDF reference (`TF-IDF_Baseline.ipynb`): micro F1 0.591, macro F1 0.517 at 0.5.

In [ ]:
TFIDF_TEST = {"micro_f1": 0.591, "macro_f1": 0.517}   # from TF-IDF_Baseline.ipynb

def run_score(s):    # validation score used to choose the winner
    if WINNER_METRIC == "contract_macro_f1":
        return s.get("val_contract", {}).get("macro_f1", float("-inf"))     # runs from before this metric existed drop out
    return s["best_val"][WINNER_METRIC]

best_tag = max(summaries, key=lambda t: run_score(summaries[t]))   # winner: validation only
rows = []
for tag, s in summaries.items():
    run = RUNS[tag]
    test = s.get("test_at_best_epoch", {})       # only present for runs whose test set was scored
    rows.append({"run": tag, "model": run["model"], "data": run.get("data", "orig"), "keep": run.get("keep", 1.0),
                 "loss": f"weighted p{run['power']} cap{run['cap']}" if run.get("power") is not None else "bce",
                 "best_epoch": s["best_epoch"],
                 "val_macro_ap": s["best_val"]["macro_ap"], "val_macro_f1": s["best_val"]["macro_f1"],
                 "val_micro_f1": s["best_val"]["micro_f1"], "val_contract_macro_f1": s.get("val_contract", {}).get("macro_f1"),
                 "test_micro_f1": test.get("micro_f1"), "test_macro_f1": test.get("macro_f1"),
                 "test_contract_macro_f1": s.get("test_contract", {}).get("macro_f1"),
                 "train_minutes": s["train_minutes"], "chosen": tag == best_tag})
comparison = pd.DataFrame(rows).assign(_score=lambda d: [run_score(summaries[t]) for t in d["run"]]).sort_values("_score", ascending=False).drop(columns="_score")
comparison = pd.concat([comparison, pd.DataFrame([{"run": "tf-idf baseline", "test_micro_f1": TFIDF_TEST["micro_f1"],
                                                    "test_macro_f1": TFIDF_TEST["macro_f1"]}])]).set_index("run")
comparison.to_csv(f"{RESULTS_DIR}/model_comparison.csv")

print(f"Chosen on validation {WINNER_METRIC}: {best_tag} ({summaries[best_tag]['model']}), epoch {summaries[best_tag]['best_epoch']}, "
      f"checkpoint: {summaries[best_tag]['checkpoint']}")
if MODE == "smoke":
    print("SMOKE RUN: the comparison only shows the pipeline works. Re-run on a GPU for real results.")
comparison.round(3)

### Weakest categories on validation (chosen model)
Focus areas for Task #3.

In [ ]:
per_cat = pd.read_csv(f"{RESULTS_DIR}/{best_tag}/val_per_category.csv", index_col=0)
per_cat.sort_values("val_ap").head(10).round(3)

### Score the winner on the test set (once)
Do this only when you are done experimenting. Set `FINAL_RUN` in the settings cell to the run chosen above, then run the cell below. It scores the saved best checkpoint on the test set once and writes `test_predictions.parquet`. Re-run the comparison cell above to see the test scores next to the validation scores.

In [ ]:
if FINAL_RUN is None:
    print(f"FINAL_RUN is not set. Chosen on validation {WINNER_METRIC}: {best_tag}. "
          f'When experiments are done, set FINAL_RUN = "{best_tag}" in the settings cell and run this cell.')
else:
    assert FINAL_RUN == best_tag, f"FINAL_RUN ({FINAL_RUN}) is not the validation winner ({best_tag})"
    score_test_once(FINAL_RUN)

## 8. Files saved
| File | In git | Contents |
|---|---|---|
| `results/training/<run>/<model>/epoch_history.csv` | Yes | Loss and validation scores per epoch |
| `results/training/<run>/<model>/summary.json` | Yes | Best epoch, validation scores, loss settings (and test scores for the final run) |
| `results/training/<run>/<model>/val_per_category.csv` | Yes | Validation AP per category (for Task #3) |
| `results/training/<run>/<model>/test_predictions.parquet` | Yes | `title`, `chunk_id`, 41 probabilities per test chunk (for Task #4; written when the test set is scored) |
| `results/training/<run>/model_comparison.csv`, `loss_curves.png` | Yes | Comparison (marks the chosen model) and loss plot |
| `models/training/<run>/<model>/best/` | No (~0.5 GB) | Best checkpoint, tokenizer, `training_info.json` |

On Colab these are under `MyDrive/Accenture-1H-contract-review-challenge/`. Smoke runs go to `models/training_smoke/`. The next cell downloads the results as a zip.

In [ ]:
if IN_COLAB and MODE == "full":
    archive = shutil.make_archive("/content/results_training", "zip", RESULTS_DIR)
    from google.colab import files
    files.download(archive)
    print("Downloading", archive, f"- unzip it into results/training/{RUN_NAME}/ in the repo and commit.")
else:
    print("Nothing to download here." if not IN_COLAB else "Smoke run: nothing worth downloading.")

## 9. Results and limitations (needs to be changed)
**Results (A100, 15 epochs).** Legal-BERT is chosen: validation macro AP 0.593 vs 0.542 for RoBERTa. At threshold 0.5 it reaches test micro F1 **0.703** and macro F1 **0.569**, vs 0.591 / 0.517 for TF-IDF. Validation loss bottoms out around epoch 11 and AP barely changes after, so 11–15 epochs is enough. Files are in `results/training/15ep/`.

**Limitations**
- Legal-BERT still loses under 0.5% of tokens to truncation; re-chunking with its tokenizer would fix this.
- Fixed 0.5 threshold and no class weights, so rare categories score low (Task #3).
- One seed and one learning rate: differences of about ±0.01 may be noise.
- TF-IDF trained on all 408 contracts; the transformers on 347 (61 held out for validation).
- Price Restrictions has no test positives, so it's excluded from test averages.

## 10. Decision Threshold for Legal Bert and Roberta



I will first get the best checkpoint of Legal-Bert

In [ ]:
legal_bert_checkpoint = f"{MODELS_DIR}/legal-bert/best"
legal_bert_threshold_model = AutoModelForSequenceClassification.from_pretrained(legal_bert_checkpoint)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(legal_bert_checkpoint)
threshold_args = training_args(f"{legal_bert_checkpoint}/threshold_eval")
threshold_args.eval_strategy = "no"
threshold_trainer = Trainer(
    model=legal_bert_threshold_model,
    args=threshold_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

I will now run the model on validation dataset to get predictions, then convert to probabilities.

In [ ]:
legal_bert_splits = prepare_splits("legal-bert")

val_output = threshold_trainer.predict(
    legal_bert_splits["validation"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)

val_probs = sigmoid(val_output.predictions)
val_labels = val_output.label_ids

print("Probabilities shape:", val_probs.shape)
print("Labels shape:", val_labels.shape)

Tuning the 41 thresholds now in the validation set


In [ ]:
def tuning_threshold(val_probs, val_labels):
  thresholds = []
  results = []

  for i, category in enumerate(CATEGORIES):
      y_true = val_labels[:, i].astype(int)
      y_prob = val_probs[:, i]

      # Some categories may not have both positive and negative examples
      if len(np.unique(y_true)) < 2:
          best_threshold = 0.5
          best_f1 = f1_score(
              y_true, (y_prob >= 0.5).astype(int), zero_division=0
          )
      else:
          precision, recall, candidates = precision_recall_curve(
              y_true, y_prob
          )

          f1_scores = (
              2 * precision[:-1] * recall[:-1]
              / (precision[:-1] + recall[:-1] + 1e-8)
          )

          best_idx = np.argmax(f1_scores)
          best_threshold = float(candidates[best_idx])
          best_f1 = float(f1_scores[best_idx])

      baseline_f1 = f1_score(
          y_true, (y_prob >= 0.5).astype(int), zero_division=0
      )

      thresholds.append(best_threshold)
      results.append({
          "category": category,
          "threshold": best_threshold,
          "f1_at_0.5": baseline_f1,
          "f1_at_tuned_threshold": best_f1,
          "positive_examples": int(y_true.sum())
      })

  threshold_results = pd.DataFrame(results)

  display(
      threshold_results.sort_values(
          "f1_at_tuned_threshold", ascending=True
      ).round(3)
  )
  return threshold_results


In [ ]:
threshold_results = tuning_threshold(val_probs, val_labels)

In [ ]:
print("Mean per-category F1 at 0.5:",
      threshold_results["f1_at_0.5"].mean().round(3))

print("Mean per-category F1 with tuned thresholds:",
      threshold_results["f1_at_tuned_threshold"].mean().round(3))

Now onto the test set where we see if f1 scores got higher

In [ ]:
test_output = threshold_trainer.predict(
    legal_bert_splits["test"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)
test_probs = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)

In [ ]:
print("Probabilities shape:", test_probs.shape)
print("Labels shape:", test_labels.shape)

We have to make sure the categories are in the same order

In [ ]:

tuned_thresholds = (
    threshold_results.set_index("category")
    .loc[CATEGORIES, "threshold"]
    .to_numpy()
)

Make predictions using both approaches of 0.5 vs tuned

In [ ]:

test_preds_05 = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= tuned_thresholds).astype(int)


In [ ]:
print("Test macro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="macro", zero_division=0), 3))

print("Test macro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="macro", zero_division=0), 3))

print("Test micro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="micro", zero_division=0), 3))

print("Test micro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="micro", zero_division=0), 3))

We see that by tuning the thresholds, it brought up macro F1 (0.555 -> 0.593) but it lowered the micro F1 (0.703 -> 0.575) for Legal Bert.


Now I will so the same thing for Roberta

In [ ]:
roberta_checkpoint = f"{MODELS_DIR}/roberta/best"
roberta_threshold_model = AutoModelForSequenceClassification.from_pretrained(roberta_checkpoint)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(roberta_checkpoint)
threshold_args = training_args(f"{roberta_checkpoint}/threshold_eval")
threshold_args.eval_strategy = "no"
threshold_trainer = Trainer(
    model=roberta_threshold_model,
    args=threshold_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [ ]:
roberta_splits = prepare_splits("roberta")

val_output = threshold_trainer.predict(
    roberta_splits["validation"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)

val_probs = sigmoid(val_output.predictions)
val_labels = val_output.label_ids

print("Probabilities shape:", val_probs.shape)
print("Labels shape:", val_labels.shape)

In [ ]:
threshold_results = tuning_threshold(val_probs, val_labels)

tuned_thresholds = (
    threshold_results.set_index("category")
    .loc[CATEGORIES, "threshold"]
    .to_numpy()
)

In [ ]:
test_output = threshold_trainer.predict(
    roberta_splits["test"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)
test_probs = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)

In [ ]:
test_preds_05 = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= tuned_thresholds).astype(int)
print("Test macro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="macro", zero_division=0), 3))

print("Test macro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="macro", zero_division=0), 3))

print("Test micro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="micro", zero_division=0), 3))

print("Test micro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="micro", zero_division=0), 3))

We can observe a similar pattern with the roberta (Macro 0.51->0.541 and Micro 0.692->0.563). Tuned thresholds improved average performance across categories, but reduced overall micro F1. Will proceed to now focus on Legal Bert and do weighted loss experiments.

## 11. Weighted Legal-BERT
The weighted-loss run is configured in the settings cell (`legal-bert-weighted`: power 0.5, cap 10) and trained in section 6 like every other run. The table below compares it with plain Legal-BERT.

In [ ]:
w_res, w_model = f"{RESULTS_DIR}/legal-bert-weighted", f"{MODELS_DIR}/legal-bert-weighted/best"
print(w_res, os.listdir(w_res)); print(w_model, os.listdir(w_model))

rows = {}
for name in ["legal-bert", "legal-bert-weighted"]:
    s = json.load(open(f"{RESULTS_DIR}/{name}/summary.json"))
    v, t = s["best_val"], s["test_at_best_epoch"]
    rows[name] = {"best_epoch": s["best_epoch"], "val_macro_ap": v["macro_ap"], "val_macro_f1": v["macro_f1"],
                  "val_micro_f1": v["micro_f1"], "test_macro_ap": t["macro_ap"],
                  "test_macro_f1": t["macro_f1"], "test_micro_f1": t["micro_f1"]}
pd.DataFrame(rows).T.round(3)

## 12. Decision Threshold for Weighted Legal Bert Model

In [ ]:
legal_bert_weighted_checkpoint = f"{MODELS_DIR}/legal-bert-weighted/best"
legal_bert_weighted_threshold_model = AutoModelForSequenceClassification.from_pretrained(legal_bert_weighted_checkpoint)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(legal_bert_weighted_checkpoint)
threshold_args = training_args(f"{legal_bert_weighted_checkpoint}/threshold_eval")
threshold_args.eval_strategy = "no"
threshold_trainer = Trainer(
    model=legal_bert_weighted_threshold_model,
    args=threshold_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [ ]:
legal_bert_weighted_splits = prepare_splits("legal-bert")

val_output = threshold_trainer.predict(
    legal_bert_weighted_splits["validation"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)

val_probs = sigmoid(val_output.predictions)
val_labels = val_output.label_ids

print("Probabilities shape:", val_probs.shape)
print("Labels shape:", val_labels.shape)

In [ ]:
threshold_results = tuning_threshold(val_probs, val_labels)

In [ ]:
print("Mean per-category F1 at 0.5:",
      threshold_results["f1_at_0.5"].mean().round(3))

print("Mean per-category F1 with tuned thresholds:",
      threshold_results["f1_at_tuned_threshold"].mean().round(3))

In [ ]:
test_output = threshold_trainer.predict(
    legal_bert_weighted_splits["test"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)
test_probs = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)

In [ ]:
print("Probabilities shape:", test_probs.shape)
print("Labels shape:", test_labels.shape)

In [ ]:

tuned_thresholds = (
    threshold_results.set_index("category")
    .loc[CATEGORIES, "threshold"]
    .to_numpy()
)

In [ ]:

test_preds_05 = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= tuned_thresholds).astype(int)


In [ ]:
print("Test macro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="macro", zero_division=0), 3))

print("Test macro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="macro", zero_division=0), 3))

print("Test micro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="micro", zero_division=0), 3))

print("Test micro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="micro", zero_division=0), 3))

The decision threshold makes it worse. We will go without the threshold tuning for the final legal-bert model.
